# 02_recipe_embedding

Develop rag pipeline: Embedd each recipe and store

Severless Rag on AWS Example: 
https://dgallitelli95.medium.com/serverless-rag-on-aws-amazon-bedrock-and-amazon-s3-vectors-8dc1f36ef5bc

Steps:
1. Create index (one-time setup)
2. Load tracker (what’s already embedded)
3. Parse JSON recipes
4. For each recipe:
    check tracker
    build text
    embed
    store (index document)
    update tracker

In [1]:
import boto3
import json
import os

In [2]:
s3 = boto3.client("s3")
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
vector_client = boto3.client("s3vectors", region_name="us-east-1")
track_file = "embedded_ids.json"

# Create Embedding Functions

In [93]:
def embed_text(text):
    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "inputText": text,
            "dimensions": 1024,
            "normalize": True
        })
    )

    result = json.loads(response["body"].read())
    return result["embedding"]

In [94]:
def create_vector_index(bucket_name, index_name, region = "us-east-1"):
    client = boto3.client("s3vectors", region_name=region)

    client.create_index(
        vectorBucketName=bucket_name,  
        indexName=index_name,
        dataType="float32",            
        dimension=1024,
        distanceMetric="cosine"         
    )

In [95]:
def index_document(bucket_name, index_name, recipe_id, embedding, metadata):
    """Writes embedding and metadata into the vector database"""
    client.put_vectors(
        bucketName=bucket_name,
        indexName=index_name,
        vectors=[{
            "id": recipe_id,
            "vector": embedding,
            "metadata": metadata
        }]
    )

In [3]:
def load_tracker(path="embedded_ids.json"):
    if not os.path.exists(path):
        with open(path, "w") as f:
            json.dump({}, f, indent=4)

    with open(path, "r") as f:
        return json.load(f)

In [97]:
def save_tracker(tracker):
    with open(track_file, "w") as f:
        json.dump(tracker, f, indent=4)

In [98]:
def remove_null_from_dict(dict):
    cleaned_dict = {}
    
    for k, v in dict.items():
        if v is None:
            continue
    
        if isinstance(v, list) and len(v) == 0:
            continue
    
        cleaned_dict[k] = v
    
    return cleaned_dict

In [99]:
def embed_recipes(data, bucket_name, index_name):

    tracker = load_tracker()
    recipes = data["recipes"]

    # -----------------------------
    # CLEAN BOOK DATA FIRST
    # -----------------------------
    attributes = remove_null_from_dict(data.get("attributes", {}))
    book_meta = remove_null_from_dict(data.get("metadata", {}))

    book_id = attributes.get("book_id")
    book_title = book_meta.get("book_title")
    book_region = attributes.get("book_region")
    book_ethnic_group = attributes.get("book_ethnic_group")
    book_period = attributes.get("book_historic_period")

    print(f"Embedding book: {book_title} ({book_id})")
    print(f"Total recipes: {len(recipes)}")

    for recipe_id, recipe in recipes.items():

        if recipe_id in tracker:
            continue

        # -----------------------------
        # BUILD SAFE EMBEDDING TEXT
        # -----------------------------
        text_parts = []

        if book_title:
            text_parts.append(f"Book Title: {book_title}")
        if book_region:
            text_parts.append(f"Region: {book_region}")
        if book_ethnic_group:
            text_parts.append(f"Ethnic Group: {book_ethnic_group}")
        if book_period:
            text_parts.append(f"Historic Period: {book_period}")

        text_parts.extend([
            f"Recipe Title: {recipe.get('recipe_title')}",
            f"Category: {recipe.get('recipe_category')}",
            f"Ingredients: {', '.join(recipe.get('recipe_ingredients', []))}",
            f"Instructions: {recipe.get('recipe_instructions')}"
        ])

        text = "\n".join(text_parts)

        embedding = embed_text(text)

        # -----------------------------
        # CLEAN METADATA BEFORE SENDING
        # -----------------------------
        metadata = remove_null_from_dict({
            "recipe_id": recipe_id,
            "recipe_title": recipe.get("recipe_title"),
            "category": recipe.get("recipe_category"),
            "book_id": book_id,
            "book_title": book_title,
            "region": book_region,
            "ethnic_group": book_ethnic_group,
            "historic_period": book_period,
            "s3_key": f"processed_cookbooks/{book_id.split('_')[0][4:]}.json" 
        })

        # -----------------------------
        # STORE VECTOR
        # -----------------------------
        vector_client.put_vectors(
            vectorBucketName=bucket_name,
            indexName=index_name,
            vectors=[{
                "key": recipe_id,
                "data": {"float32": embedding},
                "metadata": metadata
            }]
        )

        tracker[recipe_id] = True
        save_tracker(tracker)

        print(f"Embedded: {recipe_id}")

In [100]:
def load_s3_json(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    content = obj["Body"].read().decode("utf-8")
    return json.loads(content)

# Embed recipes

In [101]:
#delete the index
client.delete_index(
    vectorBucketName="recipe-vector-bucket",
    indexName="recipe-index"
)

#delete tracker filer
save_tracker({})

In [ ]:
#create vector bucket
client.create_vector_bucket(
    vectorBucketName="recipe-vector-bucket"
)

In [102]:
#create vector index 
create_vector_index(bucket_name = "recipe-vector-bucket",
                    index_name = "recipe-index",
                    region = "us-east-1")

In [103]:
#embed recipes

response = s3.list_objects_v2(
        Bucket= "feeding-america-historic-cookbooks",
        Prefix= "processed_cookbooks/"
    )

recipe_json_list = [
        obj["Key"]
        for obj in response.get("Contents", [])
        if obj["Key"].endswith(".json")
    ]


In [104]:
for json_file in recipe_json_list:
    data = load_s3_json("feeding-america-historic-cookbooks", json_file)

    # embed recipes from that file
    embed_recipes(data, bucket_name = "recipe-vector-bucket", index_name ="recipe-index")



Embedding book: The American Woman's Home: or, Principles of Domestic Science; being a Guide to the Formation and Maintenance of Economical, Healthful, Beautiful, and Christian Homes. (1869amwh)
Total recipes: 3
Embedded: 1869amwh_1
Embedded: 1869amwh_2
Embedded: 1869amwh_3
Embedding book: Manual For Army Cooks... (1896army)
Total recipes: 369
Embedded: 1896army_1
Embedded: 1896army_2
Embedded: 1896army_3
Embedded: 1896army_4
Embedded: 1896army_5
Embedded: 1896army_6
Embedded: 1896army_7
Embedded: 1896army_8
Embedded: 1896army_9
Embedded: 1896army_10
Embedded: 1896army_11
Embedded: 1896army_12
Embedded: 1896army_13
Embedded: 1896army_14
Embedded: 1896army_15
Embedded: 1896army_16
Embedded: 1896army_17
Embedded: 1896army_18
Embedded: 1896army_19
Embedded: 1896army_20
Embedded: 1896army_21
Embedded: 1896army_22
Embedded: 1896army_23
Embedded: 1896army_24
Embedded: 1896army_25
Embedded: 1896army_26
Embedded: 1896army_27
Embedded: 1896army_28
Embedded: 1896army_29
Embedded: 1896army_30
Emb

# Test Embeddings

In [105]:
#get the meta data for this recipe
query = "MOCK TURTLE SOUP"
query_embedding = embed_text(query)

response = client.query_vectors(
    vectorBucketName="recipe-vector-bucket",
    indexName="recipe-index",
    queryVector={"float32": query_embedding},
    topK=1,
    returnMetadata=True  
)

for v in response["vectors"]:
    meta = v.get("metadata", {})
    #print(f"\n--- MATCH (Score: {v.get('distance')}) ---")
    print("TITLE:", meta.get("recipe_title"))
    print("BOOK:", meta.get("book_title"))
    print("CATEGORY:", meta.get("recipe_category"))
    print("BOOK ID:", meta.get("book_id"))
    print("REGION:", meta.get("book_region"))
    print("ETHNIC GROUP:", meta.get("ethnic_group"))
    print("HISTORIC PERIOD:", meta.get("historic_period"))
    print("S3 KEY:", meta.get("s3_key"))

TITLE: MOCK TURTLE SOUP
BOOK: "Aunt Babette's" Cook Book. Foreign and Domestic Receipts for the Household. A valuable collection of receipts and hints for the housewife, many of which are not to be found elsewhere.
CATEGORY: None
BOOK ID: 1889aunt
REGION: None
ETHNIC GROUP: jewish
HISTORIC PERIOD: None
S3 KEY: processed_cookbooks/aunt.json


In [106]:
#get the embedding length of a recipe to confirm embedding
response = client.get_vectors(
    vectorBucketName="recipe-vector-bucket",
    indexName="recipe-index",
    keys=["1889aunt_5"],
    returnData=True,        
    returnMetadata=True      
)

for v in response["vectors"]:
    embedding = v.get("data", {}).get("float32")
    metadata = v.get("metadata", {})

    print("Embedding length:", len(embedding))
    print("Title:", metadata.get("recipe_title"))


Embedding length: 1024
Title: NOODLE SOUP
